In [1]:
# make autoreload cell
%load_ext autoreload
%autoreload 2



In [2]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# locate abbrd
abbrd_path = Path("/Users/rikhoekstra/develop/streamlit_worksheet/abbrd_minimal.parquet")
abbrd = pd.read_parquet(abbrd_path)
abbrd.head()

,fullname,geboortejaar,overlijdensjaar,beginjaar,eindjaar,functienaam,college,provincie,gedeputeerde
0,"'s Gravenweert, Johan van",NaN,NaN,1653.0,1656.0,ordinaris gedeputeerde,Gedeputeerde Staten van het Kwartier Nijmegen ...,None,True
1,"Aa, Willem van der",1620.0,1678.0,1645.0,1672.0,lid;gecommitteerde,Gecommitteerde Raden van Holland in het Zuider...,None,False
2,"Aa, Mauringh Cornelisz. van der",NaN,1662.0,1632.0,1662.0,lid;gecommitteerde,Gecommitteerde Raden van Holland in het Zuider...,None,False
3,"Aa, None van der",NaN,NaN,1671.0,NaN,lid;gecommitteerde,Ridderschap van Utrecht;Kamer ter Finantie van...,None,False
4,"Aa, Willem Jan Vranckensz. van der",1551.0,1613.0,1580.0,1613.0,bewindhebber;lid,Vroedschap van Rotterdam;VOC ter Kamer Rotterdam,None,False


In [4]:
# Candidate-pattern assignment for:
# 1) names in abbrd
# 2) unique delegates in 1610-1630

import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional reuse of project normalizer; fallback is simple lowercase cleanup.
try:
    from preprocess import clean_span as _clean_span
except Exception:
    _clean_span = None


def normalize_name(text: str) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    s = str(text)
    if _clean_span is not None:
        s = _clean_span(s)
    else:
        s = s.lower().strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s


# --- Build searchable variant index from 1705-1795 delegates_reference ---
ref_path = Path("data/delegates_reference.parquet")
ref = pd.read_parquet(ref_path)

variant_rows = []
for _, r in ref.iterrows():
    raw = r.get("pattern", "")
    variants = []
    if isinstance(raw, str) and raw.strip():
        variants.extend([v.strip() for v in raw.split(";") if v.strip()])
    full = r.get("fullname", "")
    if isinstance(full, str) and full.strip():
        variants.append(full.strip())

    seen = set()
    for v in variants:
        vn = normalize_name(v)
        if not vn or vn in seen:
            continue
        seen.add(vn)
        variant_rows.append(
            {
                "cons_id_str": str(r.get("cons_id_str", "")),
                "fullname": r.get("fullname", ""),
                "minjaar": r.get("minjaar", np.nan),
                "maxjaar": r.get("maxjaar", np.nan),
                "variant": v,
                "variant_norm": vn,
            }
        )

variant_df = pd.DataFrame(variant_rows)
print(f"Built variant index: {len(variant_df):,} variants from {ref['cons_id_str'].nunique():,} delegates")

# Dual TF-IDF (same spirit as project matcher)
vec_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True)
vec_word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=True)
mat_char = vec_char.fit_transform(variant_df["variant_norm"]) 
mat_word = vec_word.fit_transform(variant_df["variant_norm"]) 


def propose_patterns(query_df: pd.DataFrame, name_col: str, top_k: int = 5, min_score: float = 0.10) -> pd.DataFrame:
    q = query_df.copy()
    q = q[q[name_col].notna()].copy()
    q["query_name"] = q[name_col].astype(str)
    q["query_norm"] = q["query_name"].map(normalize_name)
    q = q[q["query_norm"].str.len() > 0].reset_index(drop=True)

    if q.empty:
        return pd.DataFrame()

    q_char = vec_char.transform(q["query_norm"]) 
    q_word = vec_word.transform(q["query_norm"]) 
    sim = 0.6 * cosine_similarity(q_char, mat_char) + 0.4 * cosine_similarity(q_word, mat_word)

    out_rows = []
    for i in range(sim.shape[0]):
        row = sim[i]
        order = np.argsort(-row)

        added = 0
        seen_delegate = set()
        for j in order:
            score = float(row[j])
            if score < min_score:
                break

            cand = variant_df.iloc[j]
            cid = cand["cons_id_str"]
            if cid in seen_delegate:
                continue

            seen_delegate.add(cid)
            out_rows.append(
                {
                    "query_name": q.loc[i, "query_name"],
                    "query_norm": q.loc[i, "query_norm"],
                    "cand_cons_id_str": cid,
                    "cand_fullname": cand["fullname"],
                    "matched_variant": cand["variant"],
                    "score": round(score, 4),
                    "cand_minjaar": cand["minjaar"],
                    "cand_maxjaar": cand["maxjaar"],
                }
            )
            added += 1
            if added >= top_k:
                break

    return pd.DataFrame(out_rows).sort_values(["query_name", "score"], ascending=[True, False])


# --- 1) ABBRD names -> possible delegate patterns ---
abbrd_name_candidates = [
    "naam", "name", "fullname", "kanonieke_naam", "delegate", "person", "person_name", "tag_text"
]
abbrd_name_col = next((c for c in abbrd_name_candidates if c in abbrd.columns), None)
if abbrd_name_col is None:
    raise ValueError(f"Could not find a name column in abbrd. Available columns: {list(abbrd.columns)}")

abbrd_unique = abbrd[[abbrd_name_col]].drop_duplicates().rename(columns={abbrd_name_col: "name"})
abbrd_matches = propose_patterns(abbrd_unique, "name", top_k=5, min_score=0.10)
print(f"abbrd name column: {abbrd_name_col}")
print(f"abbrd unique names: {len(abbrd_unique):,}")
print(f"abbrd proposed matches: {len(abbrd_matches):,}")
display(abbrd_matches.head(30))


# --- 2) 1610-1630 unique delegates -> possible delegate patterns ---
d1610_path = Path("/Users/rikhoekstra/develop/republic_delegates_data/1610_1630/consolidated/delegates_1610_1630_v1.0-rc1.parquet")
if d1610_path.exists():
    d1610 = pd.read_parquet(d1610_path)

    # Build one query string per unique delegate (prefer canonical name, fallback NAAM)
    keep_cols = [c for c in ["unified_id", "kanonieke_naam", "NAAM"] if c in d1610.columns]
    d1610u = d1610[keep_cols].copy().drop_duplicates()
    d1610u["name"] = d1610u.get("kanonieke_naam", "").fillna("").astype(str).str.strip()

    if "NAAM" in d1610u.columns:
        fallback = d1610u["name"].eq("")
        d1610u.loc[fallback, "name"] = d1610u.loc[fallback, "NAAM"].fillna("").astype(str).str.strip()

    d1610u = d1610u[d1610u["name"].str.len() > 0].copy()
    d1610_matches = propose_patterns(d1610u, "name", top_k=5, min_score=0.10)

    # Attach unified_id if available
    if "unified_id" in d1610u.columns and not d1610_matches.empty:
        d1610_matches = d1610_matches.merge(
            d1610u[["unified_id", "name"]].drop_duplicates(),
            left_on="query_name",
            right_on="name",
            how="left",
        ).drop(columns=["name"])

    print(f"1610-1630 unique delegates: {len(d1610u):,}")
    print(f"1610-1630 proposed matches: {len(d1610_matches):,}")
    display(d1610_matches.head(30))
else:
    print(f"1610-1630 file not found: {d1610_path}")

# Optional exports
# abbrd_matches.to_parquet('abbrd_possible_patterns.parquet', index=False)
# d1610_matches.to_parquet('delegates_1610_possible_patterns.parquet', index=False)

Built variant index: 2,555 variants from 1,027 delegates
abbrd name column: fullname
abbrd unique names: 8,156
abbrd proposed matches: 40,588


,query_name,query_norm,cand_cons_id_str,cand_fullname,matched_variant,score,cand_minjaar,cand_maxjaar
0,"'s Gravenweert, Johan van","'s gravenweert, johan van",18234,"Weede, Johan van","Weede, Johan van",0.4246,1688.0,1724.0
1,"'s Gravenweert, Johan van","'s gravenweert, johan van",18792,"Essen, Johan van","Essen, Johan van",0.4070,1705.0,1705.0
2,"'s Gravenweert, Johan van","'s gravenweert, johan van",16301,"Laer, Johan van","Laer, Johan van",0.3996,1718.0,1747.0
3,"'s Gravenweert, Johan van","'s gravenweert, johan van",20176,"Hoorn, Johan van","Hoorn, Johan van",0.3918,1752.0,1788.0
4,"'s Gravenweert, Johan van","'s gravenweert, johan van",18404,"Huyssen, Johan van","Huyssen, Johan van",0.3863,1736.0,1736.0
40583,", Paulus",", paulus",3058,"Paulus, Pieter",paulus,0.9981,NaN,NaN
40584,", Paulus",", paulus",17700,"Laman, Paulus","Laman, Paulus",0.6639,1777.0,1788.0
40585,", Paulus",", paulus",395,"Bosveld, Paulus","Bosveld, Paulus",0.5890,NaN,NaN
40586,", Paulus",", paulus",21106,"Aumale, Jacob Paulus d'","Aumale, Jacob Paulus d'",0.5161,1769.0,1770.0
40587,", Paulus",", paulus",18390,"Bogaard, Petrus Paulus van den","Bogaard, Petrus Paulus van den",0.4694,1749.0,1786.0


1610-1630 unique delegates: 393
1610-1630 proposed matches: 2,054


,query_name,query_norm,cand_cons_id_str,cand_fullname,matched_variant,score,cand_minjaar,cand_maxjaar,unified_id
0,"'t Hart, Gijsbert Hendricksz.","'t hart, gijsbert hendricksz.",17444,"Hart, Willem van der",hart,0.5087,1775.0,1795.0,G02337
1,"'t Hart, Gijsbert Hendricksz.","'t hart, gijsbert hendricksz.",19710,"Westenberg, Gijsbert","Westenberg, Gijsbert",0.3680,1759.0,1762.0,G02337
2,"'t Hart, Gijsbert Hendricksz.","'t hart, gijsbert hendricksz.",18576,"Sandberg, Johan Gijsbert","Sandberg, Johan Gijsbert",0.3225,1750.0,1751.0,G02337
3,"'t Hart, Gijsbert Hendricksz.","'t hart, gijsbert hendricksz.",13576,"Bruijn, Gijsbert Jan de","Bruijn, Gijsbert Jan de",0.3224,1749.0,1757.0,G02337
4,"'t Hart, Gijsbert Hendricksz.","'t hart, gijsbert hendricksz.",13583,"Patijn, Gijsbert Adriaan","Patijn, Gijsbert Adriaan",0.3221,1761.0,1788.0,G02337
5,Aelberts,aelberts,17246,"Ruever, Arend Aelbert de","Ruever, Arend Aelbert de",0.2915,1729.0,1741.0,G02122
6,Aelberts,aelberts,450,"Eldik, Aelbert Bzn. van","Eldik, Aelbert Bzn. van",0.2579,NaN,NaN,G02122
7,Aelberts,aelberts,13351,"Beelaerts, Gerard",beelaerts,0.2269,1742.0,1779.0,G02122
8,Aelberts,aelberts,15155,"Gevaerts, Johan Ockersz.",gevaerts,0.2110,1723.0,1777.0,G02122
9,Aelberts,aelberts,13651,"Gevaerts, Ocker Johansz.",gevaerts,0.2110,1766.0,1795.0,G02122


In [5]:
# Fallback regex pattern generator for names with no example-pattern candidates
# Uses constants from namenindex plus preprocess normalization rules.

import re
import importlib.util


def _load_namenindex_common_constants() -> dict[str, set[str]]:
    common_path = Path("/Users/rikhoekstra/develop/namenindex/names/common.py")
    out = {
        "TUSSENVOEGSELS": set(),
        "PREFIXES": set(),
        "POSTFIXES": set(),
        "VOORVOEGSELS": set(),
        "TERRITORIALE_TITELS": set(),
        "ROMANS": set(),
    }
    if not common_path.exists():
        return out

    spec = importlib.util.spec_from_file_location("namenindex_common", common_path)
    if spec is None or spec.loader is None:
        return out

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    for key in out:
        vals = getattr(module, key, [])
        out[key] = {str(v).strip().lower() for v in vals if str(v).strip()}
    return out


def _load_interpositions_from_preprocess() -> set[str]:
    try:
        from preprocess import _INTERP_NORMS

        toks = set()
        for _, repl in _INTERP_NORMS:
            for t in str(repl).lower().split():
                if re.fullmatch(r"[a-z]+'?[a-z]*|[a-z]+", t):
                    toks.add(t)
        return toks
    except Exception:
        return set()


COMMON = _load_namenindex_common_constants()
PREPROCESS_TUSSENVOEGSELS = _load_interpositions_from_preprocess()

TUSSENVOEGSELS = set(COMMON["TUSSENVOEGSELS"]) | PREPROCESS_TUSSENVOEGSELS
PREFIXES = set(COMMON["PREFIXES"]) | set(COMMON["VOORVOEGSELS"])
POSTFIXES = set(COMMON["POSTFIXES"])
TITLES = set(COMMON["TERRITORIALE_TITELS"])
ROMANS = set(COMMON["ROMANS"])

# Small normalization cleanup for OCR-ish punctuation variants.
PREFIXES |= {p.replace(".", "") for p in PREFIXES}
POSTFIXES |= {p.replace(".", "") for p in POSTFIXES}

print(
    "Loaded constants:",
    f"tussenvoegsels={len(TUSSENVOEGSELS)}",
    f"prefixes={len(PREFIXES)}",
    f"postfixes={len(POSTFIXES)}",
    f"titles={len(TITLES)}",
    f"romans={len(ROMANS)}",
)


def _tokenize_name(name: str) -> list[str]:
    s = normalize_name(name)
    s = re.sub(r"[^a-z0-9\s'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return [t for t in s.split(" ") if t]


def _token_to_regex(tok: str) -> str:
    # Robustify historical orthography: i/j and y/ij alternation.
    t = re.escape(tok)
    t = t.replace("ij", "(?:ij|y)")
    t = t.replace("y", "(?:y|ij)")
    t = t.replace("i", "(?:i|j)")
    t = t.replace("j", "(?:j|i)")
    return t


def _clean_tokens(tokens: list[str]) -> list[str]:
    cleaned = []
    for t in tokens:
        if t in PREFIXES:
            continue
        if t in POSTFIXES:
            continue
        if t in TITLES:
            continue
        if t in ROMANS:
            continue
        cleaned.append(t)
    return cleaned


def generate_name_regex_patterns(name: str) -> list[str]:
    tokens_raw = _tokenize_name(name)
    tokens = _clean_tokens(tokens_raw)
    if not tokens:
        return []

    toks = [_token_to_regex(t) for t in tokens]
    patterns = []

    # 1) Strict ordered full-name pattern
    patterns.append(r"\b" + r"\s+".join(toks) + r"\b")

    # 2) Flexible separators
    if len(toks) > 1:
        patterns.append(r"\b" + r"(?:\s+|[-,]\s*)".join(toks) + r"\b")

    # 3) Surname with optional tussenvoegsel chain
    surname_idx = None
    for i in range(len(tokens) - 1, -1, -1):
        if tokens[i] not in TUSSENVOEGSELS:
            surname_idx = i
            break

    if surname_idx is not None:
        surname_rx = toks[surname_idx]
        tv_prefix = [t for t in tokens[:surname_idx] if t in TUSSENVOEGSELS]
        if tv_prefix:
            tv_rx = r"(?:" + r"|".join(sorted({_token_to_regex(t) for t in tv_prefix})) + r")"
            patterns.append(r"\b(?:" + tv_rx + r"\s+){0," + str(len(tv_prefix)) + r"}" + surname_rx + r"\b")
        else:
            patterns.append(r"\b" + surname_rx + r"\b")

    # 4) Initial + surname
    non_tv = [t for t in tokens if t not in TUSSENVOEGSELS]
    if len(non_tv) >= 2:
        initial = re.escape(non_tv[0][0])
        surname_rx = _token_to_regex(non_tv[-1])
        patterns.append(r"\b" + initial + r"\.?\s+" + surname_rx + r"\b")

    # 5) Surname with optional trailing postfix (jr/sr/etc)
    if POSTFIXES and surname_idx is not None:
        suffix_rx = r"(?:" + r"|".join(sorted({_token_to_regex(p) for p in POSTFIXES})) + r")"
        patterns.append(r"\b" + _token_to_regex(tokens[surname_idx]) + r"(?:\s+" + suffix_rx + r")?\b")

    # De-duplicate while preserving order
    seen = set()
    unique = []
    for p in patterns:
        if p not in seen:
            seen.add(p)
            unique.append(p)
    return unique


def fallback_regex_for_unmatched(unique_names: pd.DataFrame, matches_df: pd.DataFrame, name_col: str) -> pd.DataFrame:
    matched_names = set(matches_df["query_name"].dropna().astype(str).unique()) if len(matches_df) else set()
    all_names = unique_names[name_col].dropna().astype(str).unique().tolist()
    unmatched = [n for n in all_names if n not in matched_names]

    rows = []
    for n in unmatched:
        pats = generate_name_regex_patterns(n)
        rows.append(
            {
                "name": n,
                "name_norm": normalize_name(n),
                "n_regex_patterns": len(pats),
                "regex_patterns": pats,
            }
        )
    return pd.DataFrame(rows)


# --- Apply fallback to abbrd ---
abbrd_fallback = fallback_regex_for_unmatched(abbrd_unique, abbrd_matches, "name")
print(f"abbrd names without candidate matches: {len(abbrd_fallback):,}")
display(abbrd_fallback.head(20))

# --- Apply fallback to 1610-1630 unique delegates ---
if "d1610u" in globals() and "d1610_matches" in globals():
    d1610_fallback = fallback_regex_for_unmatched(d1610u.rename(columns={"name": "name"}), d1610_matches, "name")
    print(f"1610-1630 names without candidate matches: {len(d1610_fallback):,}")
    display(d1610_fallback.head(20))
else:
    print("Run Cell 4 first to populate d1610u and d1610_matches.")

# Optional exports
# abbrd_fallback.to_parquet('abbrd_fallback_regex_patterns.parquet', index=False)
# d1610_fallback.to_parquet('delegates_1610_fallback_regex_patterns.parquet', index=False)

Loaded constants: tussenvoegsels=19 prefixes=18 postfixes=12 titles=18 romans=20
abbrd names without candidate matches: 0


""


1610-1630 names without candidate matches: 1


,name,name_norm,n_regex_patterns,regex_patterns
0,"N.N., ?","n.n., ?",5,"[\bn\s+n\b, \bn(?:\s+|[-,]\s*)n\b, \bn\b, \bn\..."


In [6]:
# ── Track B: Dutch Soundex reranker ──────────────────────────────────────────
#
# Implements soundex_nl() (rules documented in PLAN.md Track B section) and
# uses it to rerank the top-k TF-IDF candidates returned by match_ner().
#
# Three modes evaluated side by side:
#   A) TF-IDF only             (current production)
#   B) TF-IDF + soundex boost  (add phonetic similarity bonus to combined score)
#   C) soundex pre-filter      (keep only candidates with matching soundex key,
#                               then rank by TF-IDF score)
#
# Evaluated on the real held-out variants from Part 1 (test_real must exist).

import re
import sys
import pathlib
import importlib.util
import pandas as pd
import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd()))
from match import build_store, match_ner
from preprocess import clean_span, normalize_interpositions

# ── 1. soundex_nl implementation ─────────────────────────────────────────────
# Try to load from namenindex if available; otherwise use the built-in rules.

def _make_builtin_soundex():
    """Standalone Dutch soundex based on PLAN.md Track B description."""
    _PREFIX = re.compile(
        r"^(?:'s[-\s]|'t\s|de\s|den\s|der\s|van\s(?:de[rn]?\s)?|ten\s|d'|l')",
        re.IGNORECASE,
    )
    _STEPS = [
        # spelling normalisation (order matters)
        (re.compile(r'\bszoon\b',       re.I), 'sz'),
        (re.compile(r'naar\b',          re.I), 'na'),
        (re.compile(r'ecque\b',         re.I), 'ek'),
        (re.compile(r'eille\b',         re.I), 'elle'),
        (re.compile(r'eij|ey\b',        re.I), 'i'),
        (re.compile(r'ij|y',            re.I), 'i'),
        (re.compile(r'ouw|auw|au|ou',   re.I), 'o'),
        (re.compile(r'ck|cks|kk',       re.I), 'k'),
        (re.compile(r'(?<=[a-z])sch',   re.I), 's'),   # medial sch
        (re.compile(r'ph|ff',           re.I), 'f'),
        (re.compile(r'(?<=[ng])gh?|ngh|gg', re.I), 'g'),
        (re.compile(r'ch',              re.I), 'g'),
        (re.compile(r'v|w',             re.I), 'f'),
        (re.compile(r'(?<=[bcdfghjklmnpqrstvxz])en$', re.I), ''),  # final -en
        (re.compile(r'(?<=[bcdfghjklmnpqrstvxz])e$',  re.I), ''),  # final -e
    ]

    def soundex_token(tok: str) -> str:
        s = tok.lower().strip()
        s = _PREFIX.sub('', s)
        for pat, repl in _STEPS:
            s = pat.sub(repl, s)
        # collapse double consonants
        s = re.sub(r'(.)\1+', r'\1', s)
        return s or tok.lower()

    STOP = {
        'van', 'de', 'den', 'der', 'des', 'di', 'la', 'le', 'ten', 'het',
        'tot', 'thoe', 'in', 'op', 'of', 'en', 'het', 'dr', 'mr', 'prof',
        'jhr', 'jr', 'sr',
    }

    def soundex_nl(name: str) -> list[str]:
        """Return list of soundex keys for content tokens in *name*."""
        tokens = re.split(r"[\s\-,.']+", name.lower())
        return [soundex_token(t) for t in tokens if t and t not in STOP and len(t) > 1]

    return soundex_nl


# Try namenindex first, fall back to built-in
try:
    _spec = importlib.util.find_spec("namenindex")
    if _spec:
        from namenindex.names.soundex import soundexes_nl as soundex_nl  # type: ignore
        print("Using namenindex soundex_nl")
    else:
        raise ImportError
except Exception:
    soundex_nl = _make_builtin_soundex()
    print("Using built-in soundex_nl (namenindex not installed)")


# ── 2. Build soundex index over delegates_reference ──────────────────────────

prod_ref   = pd.read_parquet('data/delegates_reference.parquet')
prod_store = build_store(prod_ref)

id_to_soundex: dict[str, frozenset[str]] = {}
for _, row in prod_ref.iterrows():
    cid   = str(row['cons_id_str'])
    parts = [str(row.get('fullname', '') or '')]
    raw   = row.get('pattern', '')
    if isinstance(raw, str):
        parts += [v.strip() for v in raw.split(';') if v.strip()]
    keys: set[str] = set()
    for p in parts:
        keys.update(soundex_nl(p))
    id_to_soundex[cid] = frozenset(keys)

print(f"Soundex index built for {len(id_to_soundex):,} delegates")
print("Sample — Heinsius keys:",
      id_to_soundex.get(str(prod_ref[prod_ref['fullname'].str.contains('Heinsius', na=False)]['cons_id_str'].iloc[0]), set()))


# ── 3. Reranker functions ─────────────────────────────────────────────────────

def _soundex_overlap(query_keys: list[str], cand_keys: frozenset[str]) -> float:
    """Jaccard-like overlap between query and candidate soundex key sets."""
    if not query_keys or not cand_keys:
        return 0.0
    q = set(query_keys)
    inter = len(q & cand_keys)
    union = len(q | cand_keys)
    return inter / union if union else 0.0


SOUNDEX_BOOST   = 0.25   # weight of soundex bonus added to TF-IDF combined score
SOUNDEX_PENALTY = 0.10   # floor: if soundex overlap == 0, subtract this from score

def rerank_with_soundex(results: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    """Mode B: boost/penalise TF-IDF scores using soundex overlap, then re-sort."""
    out = results.copy()
    for i in range(len(out)):
        row    = out.iloc[i]
        q_keys = soundex_nl(str(row['tag_text']))

        scored: list[tuple[float, str | None]] = []
        for k in range(1, top_k + 1):
            cid = row.get(f'cand_{k}')
            sc  = float(row.get(f'score_{k}', 0) or 0)
            if cid and sc > 0:
                ov   = _soundex_overlap(q_keys, id_to_soundex.get(str(cid), frozenset()))
                adj  = sc + SOUNDEX_BOOST * ov - (SOUNDEX_PENALTY if ov == 0 else 0)
                scored.append((max(adj, 0.0), cid))
            else:
                scored.append((0.0, None))

        scored.sort(key=lambda x: -x[0])
        for rank, (sc, cid) in enumerate(scored, 1):
            out.at[i, f'cand_{rank}'] = cid
            out.at[i, f'score_{rank}'] = round(sc, 4)
    return out


def filter_by_soundex(results: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    """Mode C: zero-out candidates with no soundex key overlap, keep TF-IDF ranking."""
    out = results.copy()
    for i in range(len(out)):
        row    = out.iloc[i]
        q_keys = set(soundex_nl(str(row['tag_text'])))
        if not q_keys:
            continue
        for k in range(1, top_k + 1):
            cid = row.get(f'cand_{k}')
            sc  = float(row.get(f'score_{k}', 0) or 0)
            if cid and sc > 0:
                ov = _soundex_overlap(list(q_keys), id_to_soundex.get(str(cid), frozenset()))
                if ov == 0:
                    out.at[i, f'cand_{k}'] = None
                    out.at[i, f'score_{k}'] = 0.0
    return out


# ── 4. Evaluate on real held-out variants ────────────────────────────────────

if 'test_real' not in globals():
    print("Run Part 1 cell first to populate test_real.")
else:
    match_input = test_real[['query_text', 'year']].rename(columns={'query_text': 'tag_text'})

    res_a = match_ner(prod_store, match_input, top_k=5, year_tolerance=15, min_score=0.0)
    res_b = rerank_with_soundex(res_a.copy())
    res_c = filter_by_soundex(res_a.copy())

    def accuracy(res: pd.DataFrame, gold: pd.Series, k: int = 1) -> float:
        if k == 1:
            return (res['cand_1'].fillna('').astype(str) == gold.astype(str)).mean()
        return res.apply(
            lambda r: gold.iloc[r.name] in {r.get(f'cand_{j}', '') for j in range(1, k + 1)},
            axis=1,
        ).mean()

    gold = test_real['cons_id_str'].reset_index(drop=True)

    rows = []
    for label, res in [('A — TF-IDF only', res_a), ('B — TF-IDF + soundex boost', res_b), ('C — soundex pre-filter', res_c)]:
        rows.append({
            'mode'         : label,
            'top1_accuracy': round(accuracy(res, gold, 1), 4),
            'top3_recall'  : round(accuracy(res, gold, 3), 4),
            'top5_recall'  : round(accuracy(res, gold, 5), 4),
            'mean_score_1' : round(res['score_1'].mean(), 4),
        })

    summary_b = pd.DataFrame(rows)
    print("Track B — soundex reranker comparison:")
    display(summary_b)

    # Per-divergence-band breakdown
    if 'divergence_band' in test_real.columns:
        band_rows = []
        for band in ['near_canonical', 'moderate', 'distant', 'outlier']:
            mask = (test_real['divergence_band'] == band).values
            if not mask.any():
                continue
            for label, res in [('A', res_a), ('B', res_b), ('C', res_c)]:
                sub     = res.iloc[mask].reset_index(drop=True)
                sub_gld = gold.iloc[mask].reset_index(drop=True)
                band_rows.append({
                    'band'    : band,
                    'mode'    : label,
                    'n'       : mask.sum(),
                    'top1'    : round(accuracy(sub, sub_gld, 1), 3),
                    'top3'    : round(accuracy(sub, sub_gld, 3), 3),
                })
        band_df = pd.DataFrame(band_rows)
        print("\nPer-band top-1 accuracy:")
        display(band_df.pivot(index='band', columns='mode', values='top1').reindex(
            ['near_canonical', 'moderate', 'distant', 'outlier']))


Using built-in soundex_nl (namenindex not installed)
Soundex index built for 1,027 delegates
Sample — Heinsius keys: frozenset({'heinsius', 'anthonie'})
Run Part 1 cell first to populate test_real.


## Plan: 18th-Century Reliability Benchmark — Design Log

This document records the full design process, including intermediate steps,
rejected approaches, and benchmark outcomes.

---

### Data source

rc3: `delegates_with_patterns_1705_1795_v1.0_rc3.parquet`

| column | role |
|---|---|
| `pattern` | canonical 2-form field used in the production matcher index |
| `patterns` | full historical spelling inventory (semicolon-separated) |
| `lowerpattern` | lowercased canonical forms (used as exclusion set) |

**rc3 is the production standard going forward.**

---

### Design evolution

#### Step 1 — Initial idea (abandoned): two-variant holdout

The first approach was to use only the two forms in `pattern` as both index and
query source, holding one out.  This was too narrow: most delegates have only
two canonical forms, leaving no room for genuine held-out queries.

#### Step 2 — Discovery: `pattern` ≠ `patterns`

Inspection revealed that `pattern` was rebuilt from canonical name fields during
the rc2 correction pipeline to eliminate cross-delegate contamination.  The
richer historical spelling inventory still lives in `patterns`.  Benchmarking
from `pattern` only would miss ~80 % of the real variant diversity.

**Correction:** use `patterns` (after filtering prose contamination via
`PROSE_MARKERS`) as the benchmark source.  Variants already present in
`lowerpattern` are excluded to ensure the test set is genuinely held-out.

#### Step 3 — Random 70/30 split (superseded)

A random 70/30 per-delegate split on `patterns` variants was implemented.
This produced encouraging overall results but was criticised as measuring
**variant proliferation** rather than a meaningful difficulty gradient — all
variants were treated equally regardless of how similar or different they were
from the canonical form.

#### Step 4 — Divergence-based split (current)

Each variant is assigned a **divergence score**:

> `divergence = 1 − max(SequenceMatcher.ratio(variant, cf) for cf in canonical_forms)`

This uses Python's `difflib.SequenceMatcher`, which gives a normalised
similarity ratio in [0, 1].  Divergence 0 ≈ identical to canonical;
divergence 1 ≈ nothing in common.

Four bands are defined:

| band | divergence | interpretation |
|---|---|---|
| `near_canonical` | 0–0.15 | minor spelling shift, same stem |
| `moderate` | 0.15–0.35 | abbreviated tussenvoegsel, reordered given name, etc. |
| `distant` | 0.35–0.55 | substantial orthographic divergence |
| `outlier` | 0.55+ | historically remote or noisy form |

The test set is the 30 % **most divergent** variants per delegate; the index
set is the 70 % closest to canonical.  This makes the test set genuinely hard.

#### Step 5 — Artificially constructed bands (Part 3)

To complement the real-pattern test set, Part 3 generates synthetic queries at
controlled divergence levels by applying known historical transformations to
each delegate's canonical name fields.

**Transformations included:**

| transformation | target band | rationale |
|---|---|---|
| Drop voornaam — query is tussenvoegsel+geslachtsnaam | near_canonical | First names are almost always absent in NER spans from resolutions |
| Drop voornaam AND tussenvoegsel — bare surname | moderate | Common; scribes often wrote only the surname |
| Initial + geslachtsnaam (J. van Wassenaer) | moderate | Abbreviated given name is frequent in margin annotations |
| Split composite geslachtsnaam at hyphen/space (first part) | moderate | Delegates with hyphenated surnames are often cited by one part |
| Drop tussenvoegsel (surname only without "van"/"de") | moderate | Interposition omission is common across all scribal registers |
| Historical spelling shift: ij→y, ck→k, ae→a | distant | Frequent in 17th-/early-18th-century OCR and scribal text |
| Historical spelling shift + drop voornaam | outlier | Combined: hardest case for the matcher |

**Transformation explicitly excluded:**

- **Reversed word order (surname before voornaam)** — common in notarial and
  Latin register style, but NOT observed in the Dutch Republic resolutions
  NER spans that this matcher targets.

---

### Evaluation metrics

**Primary**

- Top-1 accuracy, Top-3 recall, Top-5 recall
- Mean reciprocal rank

**Sliding scale**

- Top-1 accuracy per divergence band (`near_canonical` → `outlier`)
- Shows the cliff edge where the matcher begins to fail

**Calibration**

- Top-1 accuracy by matcher score bin (0–0.2, 0.2–0.4, …, 0.8–1.0)
- Precision at operating thresholds (0.4, 0.6, 0.8)
- Score-gap (score_1 − score_2) for correct vs incorrect matches

**Label-error filter**

Failures with `score_1 ≥ 0.6` AND `score_gap_12 ≥ 0.15` are flagged as
_suspected label errors_ — the matcher is confidently picking a single
alternative, which often indicates a data inconsistency in the rc3 `patterns`
field rather than a genuine matcher failure.

**Family-level breakdown**

- Per-family top-1 accuracy and failure count
- Focus families: Wassenaer, Lynden, Heinsius, Ablaing, Rechteren, Reede
  (large families with many delegates, high confusion risk)

---

### Known limitations and open questions

1. **Index contamination**: the production index (`data/delegates_reference.parquet`)
   is built from rc2 `pattern` (simplified canonical), not from rc3 `patterns`.
   Some test variants may differ from what is actually indexed.  Upgrading the
   index to rc3 is a pending task.

2. **Prose contamination**: `PROSE_MARKERS` filters the most obvious non-name
   strings from `patterns`, but subtle contamination may survive.

3. **Single-variant delegates**: delegates with only one variant in `patterns`
   have no real test variant — they contribute only to the index set.

4. **Divergence measure**: `SequenceMatcher.ratio` is character-level and
   length-normalised.  It does not account for token-level reordering (dropping
   a tussenvoegsel artificially inflates divergence).  A token-Jaccard measure
   would be more linguistically appropriate but is deferred to Track B.

---

### Implementation cells

| cell | content |
|---|---|
| 9 | `benchmark_source` — raw variants extracted from rc3 `patterns` |
| 10 | Part 1 — divergence scoring + 70/30 split (most-divergent-first to test set) |
| 11 | Part 2 — canonical name string generator + coverage analysis |
| 12 | Part 3 — artificially constructed queries at each divergence band |
| 13 | Evaluation — all three parts, sliding-scale summary, label-error filter |
| 14 | Family-level error analysis |

---

### Track B (deferred)

Names-package follow-up: `names/common.py`, `names/soundex.py`,
`names/similarity.py` for improved normalisation, candidate generation,
reranking, and confidence calibration.


## Revision: `pattern` vs `patterns`

The earlier feasibility check on `data/delegates_reference.parquet` was too narrow.

What happened:

- In rc2, the `pattern` column was deliberately rebuilt from canonical name fields.
- The MANIFEST states that this was done to eliminate cross-delegate contamination from the correction pipeline.
- As a result, `pattern` is now a compact canonical matching field, usually containing forms like `van wassenaer;wassenaer`.
- The richer historical spelling inventory still lives in the rc2 `patterns` column.

Implication for benchmarking:

- A realistic synthetic benchmark should be built from cleaned variants in `patterns`.
- Those benchmark queries should then be tested against the current production matcher, which still indexes the simplified canonical `pattern` field.
- This directly measures how well the current matcher recovers historically attested variants that are not explicitly present in the canonical index.

This is a better test of practical recovery than the earlier two-variant holdout idea.

In [7]:
# Build benchmark source from historical rc3 `patterns`, excluding canonical `pattern` values.
# This measures recovery of attested variants that are *not* explicitly present
# in the current production index.

RC3_PATH = Path('/Users/rikhoekstra/develop/republic_delegates_data/1705_1795/consolidated/delegates_with_patterns_1705_1795_v1.0_rc3.parquet')
rc3 = pd.read_parquet(RC3_PATH)

PROSE_MARKERS = {
    'requeste', 'verstaan', 'gedelibereert', 'goedgevonden', 'voldongen',
    'insinuatie', 'verstek', 'poene', 'weeken', 'relaas', 'kamerbewaarder',
    'salvo', 'geinsinueert', 'suppliant'
}


def split_pattern_field(value: str) -> list[str]:
    if not isinstance(value, str) or not value.strip():
        return []
    return [part.strip(' -,:') for part in value.split(';') if part.strip(' -,:')]


def looks_like_name_variant(text: str) -> bool:
    if not isinstance(text, str):
        return False
    s = re.sub(r'\s+', ' ', text).strip()
    if len(s) < 3 or len(s) > 80:
        return False
    if not re.search(r'[A-Za-zÀ-ÿ]', s):
        return False
    if any(ch.isdigit() for ch in s):
        return False

    token_count = len(re.findall(r"[A-Za-zÀ-ÿ']+(?:-[A-Za-zÀ-ÿ']+)?", s))
    if token_count == 0 or token_count > 8:
        return False

    lower = s.lower()
    if sum(marker in lower for marker in PROSE_MARKERS) >= 1 and token_count > 5:
        return False

    return True


records = []
for _, row in rc3.iterrows():
    canonical_source = row.get('lowerpattern', row.get('pattern', ''))
    canonical_variants = {
        normalize_name(v)
        for v in split_pattern_field(canonical_source)
        if normalize_name(v)
    }

    historical_seen = set()
    for raw_variant in split_pattern_field(row.get('patterns', '')):
        if not looks_like_name_variant(raw_variant):
            continue

        norm_variant = normalize_name(raw_variant)
        if not norm_variant or norm_variant in historical_seen:
            continue
        historical_seen.add(norm_variant)

        if norm_variant in canonical_variants:
            continue

        minjaar = row.get('minjaar', pd.NA)
        maxjaar = row.get('maxjaar', pd.NA)
        if pd.notna(minjaar) and pd.notna(maxjaar):
            eval_year = int(round((float(minjaar) + float(maxjaar)) / 2))
        elif pd.notna(minjaar):
            eval_year = int(minjaar)
        elif pd.notna(maxjaar):
            eval_year = int(maxjaar)
        else:
            eval_year = 1750

        records.append({
            'cons_id_str': str(row['cons_id_str']),
            'fullname': row.get('fullname', ''),
            'variant_raw': raw_variant,
            'variant_norm': norm_variant,
            'year': eval_year,
            'minjaar': minjaar,
            'maxjaar': maxjaar,
        })

benchmark_source = pd.DataFrame(records)

print(f'Benchmark source rows: {len(benchmark_source):,}')
print(f'Delegates with at least one extra historical variant: {benchmark_source["cons_id_str"].nunique():,}')
print('Top delegates by held-out variant count:')
display(
    benchmark_source.groupby(['cons_id_str', 'fullname']).size().reset_index(name='heldout_variant_count')
    .sort_values('heldout_variant_count', ascending=False)
    .head(20)
)
display(benchmark_source.head(20))

Benchmark source rows: 1,203
Delegates with at least one extra historical variant: 185
Top delegates by held-out variant count:


,cons_id_str,fullname,heldout_variant_count
126,18379,"Ablaing, Johan Daniël d'",144
89,16307,"Rouse, Lucas Gijsbert",92
45,14024,"Heinsius, Anthonie",77
65,15127,"Wassenaer, Carel Lodewijk van",63
54,14226,"Lestevenon, Daniel",61
100,16887,"Geel, Cornelis van",56
135,19809,"Reede, Godert Adriaan van",55
75,16120,"Hoorn, Hendrik Nicolaasz. van",54
78,16189,"Rechteren tot Gramsbergen, Reinhard Burchard R...",52
34,13729,"Merens, Allard",28


,cons_id_str,fullname,variant_raw,variant_norm,year,minjaar,maxjaar
0,18379,"Ablaing, Johan Daniël d'",' Ablaing - Giessenburgh,' ablaing - giessenburgh,1761,1747.0,1775.0
1,18379,"Ablaing, Johan Daniël d'",' Ablaing van Giessenburgh,' ablaing van giessenburgh,1761,1747.0,1775.0
2,18379,"Ablaing, Johan Daniël d'",'Abaing - Giessenburgh,'abaing - giessenburgh,1761,1747.0,1775.0
3,18379,"Ablaing, Johan Daniël d'",Bout d' Ablaing van Giessenburgh,bout d' ablaing van giessenburgh,1761,1747.0,1775.0
4,18379,"Ablaing, Johan Daniël d'",Bout d' Ablaing-Giessenburgh,bout d' ablaing-giessenburgh,1761,1747.0,1775.0
5,18379,"Ablaing, Johan Daniël d'",Bout d'Ablaing van Giessenburgh,bout d'ablaing van giessenburgh,1761,1747.0,1775.0
6,18379,"Ablaing, Johan Daniël d'",V' Ahaing- Giessenburgh,v' ahaing- giessenburgh,1761,1747.0,1775.0
7,18379,"Ablaing, Johan Daniël d'",V'Abaing - Giessenburgh,v'abaing - giessenburgh,1761,1747.0,1775.0
8,18379,"Ablaing, Johan Daniël d'",V'Ablaing - Giessenburgh,v'ablaing - giessenburgh,1761,1747.0,1775.0
9,18379,"Ablaing, Johan Daniël d'",V'Ablaing van Giessenburgh,v'ablaing van giessenburgh,1761,1747.0,1775.0


In [8]:
# Part 1 — train/test split on real attested patterns, stratified by divergence from canonical.
#
# For each variant in benchmark_source we compute its divergence from the delegate's canonical
# forms (lowerpattern / pattern).  Divergence = 1 − max(SequenceMatcher ratio against each
# canonical form).  This places every variant on a spectrum:
#
#   0.0 – 0.15  near-canonical   (minor spelling shift, same stem)
#   0.15 – 0.35 moderate         (different ordering, abbreviated tussenvoegsel, etc.)
#   0.35 – 0.55 distant          (substantial orthographic divergence)
#   0.55+        outlier          (very different; likely data noise or genuinely rare form)
#
# The test set is the 30 % most divergent variants per delegate (the ones hardest to recover).
# The index set is the 70 % closest to canonical (the ones the matcher has most signal for).
# This is strictly more informative than a random 70/30 split.

import random
from difflib import SequenceMatcher
from preprocess import clean_span

RNG_SEED = 42
rng      = random.Random(RNG_SEED)

# Build canonical form lookup from rc3 lowerpattern / pattern
canonical_forms_by_id: dict[str, list[str]] = {}
for _, row in rc3.iterrows():
    cid   = str(row['cons_id_str'])
    src   = row.get('lowerpattern', row.get('pattern', ''))
    forms = [normalize_name(v) for v in split_pattern_field(src) if normalize_name(v)]
    if forms:
        canonical_forms_by_id[cid] = forms


def divergence_from_canonical(variant_norm: str, canonical_forms: list[str]) -> float:
    """1 − best SequenceMatcher ratio against any canonical form.  Range [0, 1]."""
    if not canonical_forms:
        return 1.0
    best = max(
        SequenceMatcher(None, variant_norm, cf).ratio()
        for cf in canonical_forms
    )
    return round(1.0 - best, 4)


DIVERGENCE_BINS   = [0.0, 0.15, 0.35, 0.55, 1.01]
DIVERGENCE_LABELS = ['near_canonical', 'moderate', 'distant', 'outlier']

# Add divergence to benchmark_source
records_with_div = []
for _, row in benchmark_source.iterrows():
    cid    = row['cons_id_str']
    forms  = canonical_forms_by_id.get(cid, [])
    div    = divergence_from_canonical(row['variant_norm'], forms)
    band   = DIVERGENCE_LABELS[
        next(i for i, b in enumerate(DIVERGENCE_BINS[1:]) if div < b)
    ]
    records_with_div.append({**row.to_dict(), 'divergence': div, 'divergence_band': band})

benchmark_source_div = pd.DataFrame(records_with_div)

# Per-delegate split: 30 % most divergent → test, rest → index
split_records = []
for cons_id, group in benchmark_source_div.groupby('cons_id_str'):
    rows = group.sort_values('divergence', ascending=False).reset_index(drop=True)
    n_test  = max(1, round(len(rows) * 0.30))
    for i, row in rows.iterrows():
        split_records.append({
            **row.to_dict(),
            'split' : 'test' if i < n_test else 'index',
            'source': 'real_pattern',
        })

benchmark_split = pd.DataFrame(split_records)

test_real = benchmark_split[benchmark_split['split'] == 'test'].copy()
test_real['query_text'] = test_real['variant_norm'].map(clean_span)
test_real = test_real[test_real['query_text'].str.len() > 0].reset_index(drop=True)

# Summary
print(f"Split — index: {(benchmark_split['split']=='index').sum():,}  "
      f"test: {(benchmark_split['split']=='test').sum():,}")
print(f"Delegates with ≥1 test variant: {test_real['cons_id_str'].nunique():,}")

print("\nDivergence distribution across all variants:")
display(
    benchmark_source_div.groupby('divergence_band')['divergence']
    .agg(['count', 'min', 'mean', 'max'])
    .reindex(DIVERGENCE_LABELS)
    .rename_axis('divergence_band')
)
print("\nTest-set variants per delegate (top 20 by count):")
display(
    test_real.groupby(['cons_id_str', 'fullname']).size()
    .reset_index(name='test_variants')
    .sort_values('test_variants', ascending=False).head(20)
)


Split — index: 771  test: 432
Delegates with ≥1 test variant: 185

Divergence distribution across all variants:


,count,min,mean,max
divergence_band,,,,
near_canonical,348,0.0103,0.039586,0.1489
moderate,110,0.1515,0.258011,0.3429
distant,312,0.3500,0.463568,0.5484
outlier,433,0.5510,0.707879,1.0000



Test-set variants per delegate (top 20 by count):


,cons_id_str,fullname,test_variants
126,18379,"Ablaing, Johan Daniël d'",43
89,16307,"Rouse, Lucas Gijsbert",28
45,14024,"Heinsius, Anthonie",23
65,15127,"Wassenaer, Carel Lodewijk van",19
54,14226,"Lestevenon, Daniel",18
100,16887,"Geel, Cornelis van",17
135,19809,"Reede, Godert Adriaan van",16
75,16120,"Hoorn, Hendrik Nicolaasz. van",16
78,16189,"Rechteren tot Gramsbergen, Reinhard Burchard R...",16
34,13729,"Merens, Allard",8


In [9]:
# Part 2 — artificially generated patterns from canonical name fields.
# canonical_name_strings() materialises 1–3 query strings per delegate
# (full name, tussenvoegsel+surname, surname only).
# These are compared against real held-out patterns to measure generator coverage,
# then run through the same matcher to compare recovery rates.

def canonical_name_strings(row) -> list[str]:
    voornaam      = str(row.get('voornaam',      '') or '').strip()
    tussenvoegsel = str(row.get('tussenvoegsel', '') or '').strip()
    geslachtsnaam = str(row.get('geslachtsnaam', '') or '').strip()

    parts_full = [p for p in [voornaam, tussenvoegsel, geslachtsnaam] if p]
    parts_sur  = [p for p in [tussenvoegsel, geslachtsnaam] if p]

    seen, out = set(), []
    for parts in [parts_full, parts_sur, [geslachtsnaam]]:
        s = ' '.join(parts).strip()
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    return out


gen_records = []
for _, row in rc3.iterrows():
    minjaar = row.get('minjaar', pd.NA)
    maxjaar = row.get('maxjaar', pd.NA)
    if pd.notna(minjaar) and pd.notna(maxjaar):
        eval_year = int(round((float(minjaar) + float(maxjaar)) / 2))
    elif pd.notna(minjaar):
        eval_year = int(minjaar)
    elif pd.notna(maxjaar):
        eval_year = int(maxjaar)
    else:
        eval_year = 1750

    for gen_str in canonical_name_strings(row):
        norm_gen = normalize_name(gen_str)
        if not norm_gen:
            continue
        gen_records.append({
            'cons_id_str' : str(row['cons_id_str']),
            'fullname'    : row.get('fullname', ''),
            'variant_raw' : gen_str,
            'variant_norm': norm_gen,
            'year'        : eval_year,
            'minjaar'     : minjaar,
            'maxjaar'     : maxjaar,
            'split'       : 'test',
            'source'      : 'generated_pattern',
        })

test_generated = pd.DataFrame(gen_records)
test_generated['query_text'] = test_generated['variant_norm'].map(clean_span)
test_generated = test_generated[test_generated['query_text'].str.len() > 0].reset_index(drop=True)

# Coverage: fraction of real held-out variants per delegate matched by a generated string.
real_by_del = benchmark_source.groupby('cons_id_str')['variant_norm'].apply(set).to_dict()
gen_by_del  = test_generated.groupby('cons_id_str')['variant_norm'].apply(set).to_dict()

cov_rows = []
for cid, real_set in real_by_del.items():
    covered = real_set & gen_by_del.get(cid, set())
    cov_rows.append({
        'cons_id_str'    : cid,
        'real_variants'  : len(real_set),
        'covered_by_gen' : len(covered),
        'coverage_pct'   : round(len(covered) / len(real_set) * 100, 1) if real_set else 0,
    })

coverage_df = pd.DataFrame(cov_rows)
print(f"Generated test rows : {len(test_generated):,}")
print(f"Delegates covered   : {test_generated['cons_id_str'].nunique():,}")
print(f"Mean pattern coverage by generated strings: {coverage_df['coverage_pct'].mean():.1f} %")
display(coverage_df.sort_values('coverage_pct').head(20))
display(test_generated[['fullname', 'variant_raw', 'variant_norm', 'query_text']].head(20))


Generated test rows : 2,554
Delegates covered   : 1,027
Mean pattern coverage by generated strings: 0.8 %


,cons_id_str,real_variants,covered_by_gen,coverage_pct
0,12399,4,0,0.0
116,17534,1,0,0.0
117,17540,1,0,0.0
118,17552,2,0,0.0
119,17666,1,0,0.0
120,17844,1,0,0.0
121,17977,1,0,0.0
122,18035,7,0,0.0
123,18042,1,0,0.0
124,18234,1,0,0.0


,fullname,variant_raw,variant_norm,query_text
0,"Abbink, Bernard Engelbert",Bernard Engelbert Abbink,bernard engelbert abbink,bernard engelbert abbink
1,"Abbink, Bernard Engelbert",Abbink,abbink,abbink
2,"Ablaing, Johan Daniël d'",Johan Daniël d' Ablaing,johan daniël d' ablaing,johan daniël d' ablaing
3,"Ablaing, Johan Daniël d'",d' Ablaing,d' ablaing,d' ablaing
4,"Ablaing, Johan Daniël d'",Ablaing,ablaing,ablaing
5,"Acquet, Hendrik Georgesz. d'",Hendrik Georgesz. d' Acquet,hendrik georgesz. d' acquet,hendrik georgesz. d' acquet
6,"Acquet, Hendrik Georgesz. d'",d' Acquet,d' acquet,d' acquet
7,"Acquet, Hendrik Georgesz. d'",Acquet,acquet,acquet
8,"Alberda, Onno Reint",Onno Reint Alberda,onno reint alberda,onno reint alberda
9,"Alberda, Onno Reint",Alberda,alberda,alberda


In [10]:
# Part 3 — artificially constructed queries at controlled divergence levels.
#
# We generate synthetic queries from each delegate's canonical name fields
# (voornaam, tussenvoegsel, geslachtsnaam) by applying realistic historical
# transformations.  Each transformation targets a known divergence band so we
# can evaluate the matcher at a specific difficulty level.
#
# Transformations applied (in order of increasing divergence):
#
#   near_canonical
#     (a) Drop voornaam → tussenvoegsel + geslachtsnaam
#         Rationale: first names are almost always absent in NER spans
#
#   moderate
#     (b) Drop voornaam + tussenvoegsel → bare geslachtsnaam
#     (c) Initial-only voornaam + geslachtsnaam (e.g. "J. van Wassenaer")
#     (d) Split composite geslachtsnaam at first hyphen/space
#         (e.g. "Lynden-Hemmen" → "Lynden")
#     (e) Drop tussenvoegsel, keep voornaam + geslachtsnaam
#
#   distant
#     (f) Historical spelling: ij→y, ck→k, ae→a  (applied to geslachtsnaam)
#
#   outlier
#     (g) Historical spelling on bare geslachtsnaam (drop voornaam + TV)
#
# NOT included:
#   Reversed word order (surname before voornaam) — common in notarial /
#   Latin register style, but not observed in Dutch Republic resolution NER spans.

import re as _re

_SPELL_SHIFTS = [
    (_re.compile(r'ij'),  'y'),
    (_re.compile(r'ck'),  'k'),
    (_re.compile(r'ae'),  'a'),
    (_re.compile(r'oe'),  'u'),   # "Hoeck" → "Huck"-ish; loose but real
    (_re.compile(r'th'),  't'),   # "Thissen" → "Tissen"
    (_re.compile(r'ou'),  'u'),   # "Boudewijns" → "Budewijns"
]


def _apply_spelling_shifts(s: str) -> str:
    for pat, repl in _SPELL_SHIFTS:
        s = pat.sub(repl, s)
    return s


def _split_compound(geslachtsnaam: str) -> str | None:
    """Return the first component of a composite surname, or None if simple."""
    s = geslachtsnaam.strip()
    # hyphenated: "Lynden-Hemmen"
    if '-' in s:
        return s.split('-')[0].strip()
    # multi-word after tussenvoegsel has been removed: detect 2+ capitalised tokens
    parts = s.split()
    if len(parts) >= 2 and parts[1][0].isupper():
        return parts[0]
    return None


def _build_query(parts: list[str]) -> str:
    return normalize_name(' '.join(p for p in parts if p))


ART_TRANSFORMATIONS = [
    # (label, target_band, fn(voornaam, tv, geslacht) -> str | None)
    ('drop_voornaam',           'near_canonical',
     lambda v, t, g: _build_query([t, g]) if (t or g) else None),

    ('drop_voornaam_and_tv',    'moderate',
     lambda v, t, g: _build_query([g]) if g else None),

    ('initial_surname',         'moderate',
     lambda v, t, g: _build_query([v[0] + '.', g]) if v and g else None),

    ('split_compound',          'moderate',
     lambda v, t, g: _build_query([t, _split_compound(g)]) if g and _split_compound(g) else None),

    ('drop_tv',                 'moderate',
     lambda v, t, g: _build_query([v, g]) if v and g else None),

    ('historical_spelling',     'distant',
     lambda v, t, g: _build_query([t, _apply_spelling_shifts(g)]) if g else None),

    ('historical_bare_surname', 'outlier',
     lambda v, t, g: _build_query([_apply_spelling_shifts(g)]) if g else None),
]


art_records = []
for _, row in rc3.iterrows():
    v  = str(row.get('voornaam',      '') or '').strip()
    tv = str(row.get('tussenvoegsel', '') or '').strip()
    g  = str(row.get('geslachtsnaam', '') or '').strip()

    minjaar = row.get('minjaar', pd.NA)
    maxjaar = row.get('maxjaar', pd.NA)
    if pd.notna(minjaar) and pd.notna(maxjaar):
        eval_year = int(round((float(minjaar) + float(maxjaar)) / 2))
    elif pd.notna(minjaar):
        eval_year = int(minjaar)
    elif pd.notna(maxjaar):
        eval_year = int(maxjaar)
    else:
        eval_year = 1750

    seen_norms: set[str] = set()
    for label, band, fn in ART_TRANSFORMATIONS:
        try:
            q = fn(v, tv, g)
        except Exception:
            continue
        if not q:
            continue
        norm_q = normalize_name(q)
        if not norm_q or norm_q in seen_norms:
            continue
        seen_norms.add(norm_q)
        art_records.append({
            'cons_id_str'    : str(row['cons_id_str']),
            'fullname'       : row.get('fullname', ''),
            'variant_raw'    : q,
            'variant_norm'   : norm_q,
            'year'           : eval_year,
            'minjaar'        : minjaar,
            'maxjaar'        : maxjaar,
            'split'          : 'test',
            'source'         : 'artificial',
            'divergence_band': band,
            'transformation' : label,
        })

test_artificial = pd.DataFrame(art_records)
test_artificial['query_text'] = test_artificial['variant_norm'].map(clean_span)
test_artificial = test_artificial[test_artificial['query_text'].str.len() > 0].reset_index(drop=True)

print(f"Artificial test rows : {len(test_artificial):,}")
print(f"Delegates covered    : {test_artificial['cons_id_str'].nunique():,}")
print('\nRows per transformation:')
display(
    test_artificial.groupby(['transformation', 'divergence_band']).size()
    .reset_index(name='rows')
    .sort_values('divergence_band')
)
display(test_artificial[['fullname', 'transformation', 'variant_raw', 'variant_norm', 'divergence_band']].head(20))


Artificial test rows : 4,105
Delegates covered    : 1,027

Rows per transformation:


,transformation,divergence_band,rows
4,historical_spelling,distant,332
0,drop_tv,moderate,1025
2,drop_voornaam_and_tv,moderate,501
5,initial_surname,moderate,1026
6,split_compound,moderate,16
1,drop_voornaam,near_canonical,1027
3,historical_bare_surname,outlier,178


,fullname,transformation,variant_raw,variant_norm,divergence_band
0,"Abbink, Bernard Engelbert",drop_voornaam,abbink,abbink,near_canonical
1,"Abbink, Bernard Engelbert",initial_surname,b. abbink,b. abbink,moderate
2,"Abbink, Bernard Engelbert",drop_tv,bernard engelbert abbink,bernard engelbert abbink,moderate
3,"Ablaing, Johan Daniël d'",drop_voornaam,d' ablaing,d' ablaing,near_canonical
4,"Ablaing, Johan Daniël d'",drop_voornaam_and_tv,ablaing,ablaing,moderate
5,"Ablaing, Johan Daniël d'",initial_surname,j. ablaing,j. ablaing,moderate
6,"Ablaing, Johan Daniël d'",drop_tv,johan daniël ablaing,johan daniël ablaing,moderate
7,"Acquet, Hendrik Georgesz. d'",drop_voornaam,d' acquet,d' acquet,near_canonical
8,"Acquet, Hendrik Georgesz. d'",drop_voornaam_and_tv,acquet,acquet,moderate
9,"Acquet, Hendrik Georgesz. d'",initial_surname,h. acquet,h. acquet,moderate


In [11]:
# Evaluate Part 1 (real held-out), Part 2 (generated), and Part 3 (artificial)
# against the production matcher.
# Production store: data/delegates_reference.parquet (canonical `pattern`, rc2).

from match import build_store, match_ner

prod_ref   = pd.read_parquet('data/delegates_reference.parquet')
prod_store = build_store(prod_ref)


def run_eval(query_df: pd.DataFrame, label: str) -> pd.DataFrame:
    match_input = query_df[['query_text', 'year']].rename(columns={'query_text': 'tag_text'})
    results     = match_ner(prod_store, match_input, top_k=5, year_tolerance=15, min_score=0.0)
    ev = pd.concat([query_df.reset_index(drop=True), results.reset_index(drop=True)], axis=1)
    ev['top1_correct'] = ev['cand_1'].fillna('').astype(str) == ev['cons_id_str'].astype(str)
    ev['top3_correct'] = ev.apply(
        lambda r: r['cons_id_str'] in {r.get(f'cand_{k}', '') for k in range(1, 4)}, axis=1)
    ev['top5_correct'] = ev.apply(
        lambda r: r['cons_id_str'] in {r.get(f'cand_{k}', '') for k in range(1, 6)}, axis=1)
    ev['score_gap_12'] = ev['score_1'].fillna(0) - ev['score_2'].fillna(0)
    ev['score_bin']    = pd.cut(
        ev['score_1'], bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
    ev['part'] = label
    return ev


eval_real       = run_eval(test_real,       'real_held_out')
eval_generated  = run_eval(test_generated,  'generated')
eval_artificial = run_eval(test_artificial, 'artificial')
benchmark_eval  = pd.concat([eval_real, eval_generated, eval_artificial], ignore_index=True)

# ── Overall summary ───────────────────────────────────────────────────────────
summary = (
    benchmark_eval.groupby('part')
    .agg(
        rows          = ('cons_id_str', 'size'),
        top1_accuracy = ('top1_correct', 'mean'),
        top3_recall   = ('top3_correct', 'mean'),
        top5_recall   = ('top5_correct', 'mean'),
        mean_score_1  = ('score_1', 'mean'),
        mean_gap_12   = ('score_gap_12', 'mean'),
    ).reset_index()
)

# ── Sliding scale: accuracy by divergence band ────────────────────────────────
# Both real_held_out (Part 1) and artificial (Part 3) carry divergence_band.
div_parts = benchmark_eval[benchmark_eval['divergence_band'].notna()].copy()
divergence_summary = (
    div_parts.groupby(['part', 'divergence_band'])
    .agg(
        rows            = ('cons_id_str', 'size'),
        top1_accuracy   = ('top1_correct', 'mean'),
        top3_recall     = ('top3_correct', 'mean'),
        mean_score_1    = ('score_1', 'mean'),
        mean_gap_12     = ('score_gap_12', 'mean'),
    )
    .reset_index()
)
# Pivot: bands as columns so real vs artificial are side by side
div_pivot = divergence_summary.pivot(
    index='divergence_band', columns='part', values='top1_accuracy'
).reindex(DIVERGENCE_LABELS)

# ── Artificial: per-transformation accuracy ───────────────────────────────────
if 'transformation' in eval_artificial.columns:
    transf_summary = (
        eval_artificial.groupby(['transformation', 'divergence_band'])
        .agg(
            rows          = ('cons_id_str', 'size'),
            top1_accuracy = ('top1_correct', 'mean'),
            top3_recall   = ('top3_correct', 'mean'),
            mean_score_1  = ('score_1', 'mean'),
        )
        .reset_index()
        .sort_values('divergence_band')
    )

# ── Calibration by score bin ──────────────────────────────────────────────────
calibration = (
    benchmark_eval.groupby(['part', 'score_bin'], observed=False)
    .agg(
        rows          = ('cons_id_str', 'size'),
        top1_accuracy = ('top1_correct', 'mean'),
        top3_recall   = ('top3_correct', 'mean'),
    ).reset_index()
)

print('Summary by source:')
display(summary)
print('\nSliding scale — top-1 accuracy by divergence band:')
display(div_pivot)
print('\nPer-transformation accuracy (Part 3 — artificial):')
display(transf_summary)
print('\nCalibration by score bin:')
display(calibration)
print('\nSample failures (real held-out):')
display(
    eval_real.loc[~eval_real['top1_correct'], [
        'fullname', 'variant_raw', 'query_text', 'divergence_band',
        'cons_id_str', 'cand_1', 'score_1', 'cand_2', 'score_2',
    ]].head(25)
)

# ── Suspected label errors ────────────────────────────────────────────────────
suspected_label_errors = eval_real.loc[
    ~eval_real['top1_correct']
    & (eval_real['score_1'] >= 0.6)
    & (eval_real['score_gap_12'] >= 0.15)
].copy()

id_to_name = rc3.set_index('cons_id_str')['fullname'].to_dict()
suspected_label_errors['cand_1_fullname'] = (
    suspected_label_errors['cand_1'].map(lambda x: id_to_name.get(x, x))
)

n_failures  = (~eval_real['top1_correct']).sum()
n_suspected = len(suspected_label_errors)
pct         = n_suspected / n_failures * 100 if n_failures else 0

print(f"\nSuspected label errors: {n_suspected:,} of {n_failures:,} failures "
      f"({pct:.1f} %) — high-confidence unambiguous picks against the label")
print("Adjusted top-1 accuracy (excluding suspected label errors): "
      f"{(eval_real['top1_correct'].sum() + n_suspected) / len(eval_real):.3f}")
display(
    suspected_label_errors[[
        'fullname', 'variant_raw', 'divergence_band',
        'cons_id_str', 'cand_1', 'cand_1_fullname', 'score_1', 'score_gap_12',
    ]].sort_values('score_1', ascending=False)
)


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
# Family-level error analysis.
# Groups delegates by surname (geslachtsnaam / last word of fullname)
# and reports top-1 accuracy, recall, and typical confusions.

def extract_family_key(fullname: str) -> str:
    s = str(fullname or '').strip()
    if ',' in s:
        return s.split(',')[0].strip().lower()
    parts = s.split()
    return parts[-1].lower() if parts else s.lower()


benchmark_eval['family_key'] = benchmark_eval['fullname'].map(extract_family_key)

family_stats = (
    benchmark_eval[benchmark_eval['part'] == 'real_held_out']
    .groupby('family_key')
    .agg(
        n_delegates   = ('cons_id_str', 'nunique'),
        n_variants    = ('cons_id_str', 'size'),
        top1_accuracy = ('top1_correct', 'mean'),
        top3_recall   = ('top3_correct', 'mean'),
        mean_score_1  = ('score_1', 'mean'),
    )
    .reset_index()
)

print('Families with most failures (lowest top-1 accuracy, ≥ 3 variants):')
display(
    family_stats[family_stats['n_variants'] >= 3]
    .sort_values('top1_accuracy').head(20)
)

print('Families with most variants (top 20):')
display(family_stats.sort_values('n_variants', ascending=False).head(20))

FOCUS_FAMILIES = ['wassenaer', 'lynden', 'heinsius', 'ablaing', 'rechteren', 'reede']
focus = family_stats[family_stats['family_key'].isin(FOCUS_FAMILIES)].sort_values('top1_accuracy')
print('Focus families:')
display(focus)

print('Top confusions per focus family:')
confusion_frames = []
for fam in FOCUS_FAMILIES:
    mask = (
        (benchmark_eval['part'] == 'real_held_out')
        & (benchmark_eval['family_key'] == fam)
        & (~benchmark_eval['top1_correct'])
    )
    sub = benchmark_eval.loc[mask, [
        'fullname', 'variant_raw', 'query_text',
        'cons_id_str', 'cand_1', 'score_1', 'cand_2', 'score_2',
    ]]
    if not sub.empty:
        confusion_frames.append(sub.head(10))

if confusion_frames:
    display(pd.concat(confusion_frames, ignore_index=True))
else:
    print('No failures found in focus families.')


## Track B placeholder: names package follow-up

This remains a separate todo item.

Only after the synthetic benchmark is stable should we inspect whether `names/common.py`, `names/soundex.py`, and `names/similarity.py` improve:

- normalization
- candidate generation
- reranking
- confidence calibration